<a href="https://colab.research.google.com/github/zheng-anthony/AURA/blob/main/notebooks/AI4LL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# CELL 1: SECURE DATA INGESTION
# ==========================================
!pip install -q kaggle

import os
from google.colab import files

print("📂 UPLOAD YOUR KAGGLE.JSON FILE NOW:")
uploaded = files.upload()

# Move the Kaggle API key to the secure hidden folder
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

print("\n📡 DOWNLOADING RDD 2022 DATASET (Slide 3)...")
!kaggle datasets download -d aliabdelmenam/rdd-2022

print("\n📡 DOWNLOADING ANNOTATED POTHOLE DATASET (Slide 5)...")
!kaggle datasets download -d virenbr11/pothole-and-plain-rode-images

print("\n📦 UNZIPPING DATASETS...")
!mkdir -p /content/raw_data
!unzip -q rdd-2022.zip -d /content/raw_data/rdd
!unzip -q pothole-and-plain-rode-images.zip -d /content/raw_data/pothole

print("\n✅ DATA INGESTION COMPLETE!")

📂 UPLOAD YOUR KAGGLE.JSON FILE NOW:


Saving kaggle.json to kaggle.json

📡 DOWNLOADING RDD 2022 DATASET (Slide 3)...
Dataset URL: https://www.kaggle.com/datasets/aliabdelmenam/rdd-2022
License(s): CC-BY-SA-4.0
100% 9.90G/9.90G [01:52<00:00, 94.7MB/s]


📡 DOWNLOADING ANNOTATED POTHOLE DATASET (Slide 5)...
Dataset URL: https://www.kaggle.com/datasets/virenbr11/pothole-and-plain-rode-images
License(s): unknown
100% 241M/241M [00:01<00:00, 170MB/s]


📦 UNZIPPING DATASETS...

✅ DATA INGESTION COMPLETE!


In [ ]:
import os
import cv2
import shutil
import albumentations as A
transform = A.Compose(
    [
        A.OneOf([
            A.GaussianBlur(blur_limit=(3, 7), p=0.4),
            A.MotionBlur(blur_limit=(3, 7), p=0.3),
        ], p=0.5),

        A.OneOf([
            # Fog branch: fog + slight desaturation/brightness shift, always paired
            A.Sequential([
                A.RandomFog(
                    fog_coef_range=(0.1, 0.3),
                    alpha_coef=0.08,
                    p=1.0
                ),
                A.RandomBrightnessContrast(
                    brightness_limit=(-0.15, 0.05),
                    contrast_limit=(-0.1, 0.05),
                    p=1.0
                ),
                A.HueSaturationValue(
                    hue_shift_limit=0,
                    sat_shift_limit=(-30, -5),
                    val_shift_limit=0,
                    p=1.0
                ),
            ], p=0.4),

            # Rain branch: rain + its own darker/duller look, always paired
            A.Sequential([
                A.RandomRain(
                    drop_length=10,
                    drop_width=1,
                    blur_value=3,
                    p=1.0
                ),
                A.RandomBrightnessContrast(
                    brightness_limit=(-0.2, 0.0),
                    contrast_limit=(-0.1, 0.1),
                    p=1.0
                ),
                A.HueSaturationValue(
                    hue_shift_limit=0,
                    sat_shift_limit=(-20, 0),
                    val_shift_limit=0,
                    p=1.0
                ),
            ], p=0.3),
        ], p=0.4),
    ],
    bbox_params=A.BboxParams(
        format="yolo",
        label_fields=["class_labels"]
    ),
)

print("⚙️ INITIALIZING DATA PIPELINE (Merging Custom Splits)...")

BASE_DIR = '/content/raw_data'
OUTPUT_DIR = '/content/yolo_dataset'

# YOLO uses 'val' instead of 'test', so we map your 'test' folders to 'val'
SOURCES = {
    'train': [
        os.path.join(BASE_DIR, 'pothole/My Dataset/train'),
        os.path.join(BASE_DIR, 'rdd/RDD_SPLIT/train')
    ],
    'val': [
        os.path.join(BASE_DIR, 'pothole/My Dataset/test'),
        os.path.join(BASE_DIR, 'rdd/RDD_SPLIT/test')
    ]
}

def process_data(source_folders, split_name):
    print(f"📦 Processing {split_name} split from multiple sources...")
    target_img_dir = os.path.join(OUTPUT_DIR, split_name, 'images')
    target_lab_dir = os.path.join(OUTPUT_DIR, split_name, 'labels')
    os.makedirs(target_img_dir, exist_ok=True)
    os.makedirs(target_lab_dir, exist_ok=True)

    for folder in source_folders:
        if not os.path.exists(folder):
            print(f"⚠️ Warning: Folder not found {folder}")
            continue

        print(f"   -> Scanning {folder}...")
        # os.walk allows us to dig into subfolders in case images/labels are separated
        for root, dirs, files in os.walk(folder):
            for filename in files:
                if filename.lower().endswith(('.jpg', '.jpeg', '.png')):

                    img_path = os.path.join(root, filename)

                    # Read and resize to Slide 5 promise (600x400)
                    img = cv2.imread(img_path)
                    if img is None: continue
                    resized_img = cv2.resize(img, (600, 400))
                    cv2.imwrite(os.path.join(target_img_dir, filename), resized_img)

                    # Locate the corresponding .txt label file
                    lab_name = os.path.splitext(filename)[0] + '.txt'

                    # Check the same directory first
                    lab_src = os.path.join(root, lab_name)

                    # If not there, check if it's in a parallel 'labels' directory
                    if not os.path.exists(lab_src):
                        possible_lab_dir = root.replace('images', 'labels')
                        lab_src = os.path.join(possible_lab_dir, lab_name)

                    lab_dst = os.path.join(target_lab_dir, lab_name)

                    if os.path.exists(lab_src):
                        shutil.copy(lab_src, lab_dst)
                    else:
                        # Slide 13: Negative Sampling Injection
                        # If no label exists, create an empty txt file to teach YOLO "this is a healthy road"
                        open(lab_dst, 'w').close()

process_data(SOURCES['train'], 'train')
process_data(SOURCES['val'], 'val')

yaml_content = f"""
path: {OUTPUT_DIR}
train: train/images
val: val/images

names:
  0: D00 (Longitudinal Crack)
  1: D10 (Transverse Crack)
  2: D20 (Alligator Crack)
  3: D40 (Pothole)
"""

with open(os.path.join(OUTPUT_DIR, 'data.yaml'), 'w') as f:
    f.write(yaml_content)

print("\n✅ DATA PIPELINE FINISHED. DATASETS MERGED & NEGATIVE SAMPLES INJECTED.")

⚙️ INITIALIZING DATA PIPELINE (Merging Custom Splits)...
📦 Processing train split from multiple sources...
   -> Scanning /content/raw_data/pothole/My Dataset/train...
   -> Scanning /content/raw_data/rdd/RDD_SPLIT/train...
📦 Processing val split from multiple sources...
   -> Scanning /content/raw_data/pothole/My Dataset/test...
   -> Scanning /content/raw_data/rdd/RDD_SPLIT/test...

✅ DATA PIPELINE FINISHED. DATASETS MERGED & NEGATIVE SAMPLES INJECTED.


In [ ]:

# ==========================================
# CELL 3: AI MODEL TRAINING (YOLOv8)
# ==========================================
print("🧠 INSTALLING ULTRALYTICS NEURAL NETWORK...")
!pip install -q ultralytics

from ultralytics import YOLO
from google.colab import drive

print("📂 MOUNTING GOOGLE DRIVE FOR NATIVE CHECKPOINTING...")
drive.mount('/content/drive')

# Initialize the chosen YOLOv8 architecture (Slide 9 Promise)
model = YOLO('yolov8s.pt')

print("\n🚀 INITIATING TRAINING SEQUENCE WITH BIAS MITIGATION AUGMENTATION...")
# SLIDE 13 PROMISE: Synthetic contrast adjustments & weather overlays.
# We pass augmentation hyper-parameters directly into the training loop!
results = model.train(
    data='/content/yolo_dataset/data.yaml',
    epochs=50,             # Full training cycle
    imgsz=400,             # Internal processing size
    batch=16,
    hsv_h=0.015,           # Hue augmentation (lighting)
    hsv_s=0.7,             # Saturation augmentation (weather contrast)
    hsv_v=0.4,             # Value augmentation (simulates dark clouds/fog)
    perspective=0.0001,    # Forces learning from horizontal vanishing points (Slide 6)
    project='/content/drive/MyDrive/AI4ALL',
    name='PaveVision_v1',  # Saves directly to your Google Drive!
    save=True,             # 🟢 ENABLE NATIVE CHECKPOINT CALLBACK
    save_period=5          # 🟢 BACKUP WEIGHTS EVERY 5 EPOCHS
)

print("\n✅ TRAINING COMPLETE! WEIGHTS SAVED.")

🧠 INSTALLING ULTRALYTICS NEURAL NETWORK...
📂 MOUNTING GOOGLE DRIVE FOR NATIVE CHECKPOINTING...


ValueError: Mountpoint must not already contain files

In [ ]:
# ==========================================
# CELL 4: RIGOROUS EVALUATION (SLIDE 10)
# ==========================================
import os
import matplotlib.pyplot as plt
import cv2

print("📊 GENERATING MODEL EVALUATION METRICS...")

# The results are saved in the project directory created in Cell 3
RESULTS_DIR = '/content/drive/MyDrive/AI4ALL/PaveVision_v1'

def display_metric(image_name, title):
    img_path = os.path.join(RESULTS_DIR, image_name)
    if os.path.exists(img_path):
        img = cv2.imread(img_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(15, 10))
        plt.imshow(img_rgb)
        plt.title(title, fontsize=16, fontweight='bold')
        plt.axis('off')
        plt.show()
    else:
        print(f"⚠️ Could not find {image_name}. Make sure training finished completely.")

# 1. Display the primary F1-Score Curve (Balances Precision and Recall)
display_metric('F1_curve.png', 'Model Metric: F1-Score (Harmonic Mean)')

# 2. Display the Precision-Recall Curve
display_metric('PR_curve.png', 'Model Metric: Precision vs Recall')

# 3. Display the Confusion Matrix (Shows exactly where False Positives happen)
display_metric('confusion_matrix.png', 'Error Analysis: Confusion Matrix')

print("\n✅ EVALUATION COMPLETE. SCREENSHOT THESE FOR YOUR PRESENTATION.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')